In [2]:
pip install pandas

  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 8.5 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.7 MB 4.8 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.7 MB 4.4 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.7 MB 4.3 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.7 MB 4.2 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.7 MB 4.1 MB/s eta 0:00:02
   ------------------------ --------------- 6.0/9.7 MB 4.1 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/9.7 MB 4.0 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.7 MB 4.1 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.7 MB 4.0 MB/s eta 0:00:01
   ------------------------------------- -- 9.2/9.7 MB 4.0 MB/s eta 0:00:01
   ------------------------

In [5]:
pip install scikit-learn

  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl (8.0 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl (36.5 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ------------------

In [6]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [7]:
# --- STEP 2: LOAD DATA ---
# Load the necessary CSV files (Ensure these file names match your downloaded files)
try:
    customers = pd.read_csv('data/olist_customers_dataset.csv')
    orders = pd.read_csv('data/olist_orders_dataset.csv')
    payments = pd.read_csv('data/olist_order_payments_dataset.csv')
    reviews = pd.read_csv('data/olist_order_reviews_dataset.csv')
    print("Data loaded successfully.")
except FileNotFoundError:
    print("Error: CSV files not found. Please check file paths.")


Data loaded successfully.


In [8]:
    
# --- STEP 3: DATA CLEANING ---

# 1. Merge Tables to create a Master View
# We join Customers -> Orders -> Payments -> Reviews
df = customers.merge(orders, on='customer_id', how='left')
df = df.merge(payments, on='order_id', how='left')
df = df.merge(reviews, on='order_id', how='left')

print(f"Initial Shape after merging: {df.shape}")

# 2. Handle Missing Values
# We only want 'delivered' orders for accurate churn analysis
df = df[df['order_status'] == 'delivered']

# Drop rows with critical missing data (e.g., payment value or review score)
df = df.dropna(subset=['payment_value', 'review_score', 'order_delivered_customer_date'])

# Convert date columns to datetime format
date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date']
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print("Data Cleaning Complete.")


Initial Shape after merging: (104478, 22)
Data Cleaning Complete.


In [9]:
# --- STEP 4: FEATURE ENGINEERING (Processing) ---

# We need to aggregate data by 'customer_unique_id' (one row per customer)
# We will calculate: Recency, Frequency, Monetary, and Satisfaction

# 1. Reference Date: The last date in the dataset + 1 day (to calculate recency)
reference_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

# 2. Group by Customer and Calculate Features
customer_data = df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (reference_date - x.max()).days, # Recency
    'order_id': 'count',                                                   # Frequency
    'payment_value': 'sum',                                                # Monetary
    'review_score': 'mean'                                                 # Satisfaction
}).reset_index()

# Rename columns for clarity
customer_data.columns = ['customer_unique_id', 'recency_days', 'total_orders', 'total_spent', 'avg_review_score']



In [10]:
# --- STEP 5: CHURN LABELING (The Gap Solution) ---
# Define Churn: If a customer hasn't ordered in more than 180 days (approx 6 months)
churn_threshold = 180
customer_data['churn_label'] = np.where(customer_data['recency_days'] > churn_threshold, 1, 0)

print("Feature Engineering Complete. Sample Data:")
print(customer_data.head())

# --- STEP 6: AI MODEL (Propensity Prediction) ---
# We train a model to predict the probability of churn (0.0 to 1.0)

# Select features (X) and Target (y)
features = ['recency_days', 'total_orders', 'total_spent', 'avg_review_score']
X = customer_data[features]
y = customer_data['churn_label']

# Split data (Train/Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Random Forest Classifier
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate Model (Optional: Print report to show in assignment)
print("\n--- Model Evaluation ---")
print(classification_report(y_test, model.predict(X_test)))

# Predict Churn Probability for ALL customers
# We want the probability (0-100%), not just the 0/1 label
customer_data['churn_probability'] = model.predict_proba(X)[:, 1]



Feature Engineering Complete. Sample Data:
                 customer_unique_id  recency_days  total_orders  total_spent  \
0  0000366f3b9a7992bf8c76cfdf3221e2           112             1       141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f           115             1        27.19   
2  0000f46a3911fa3c0805444483337064           537             1        86.22   
3  0000f6ccb0745a6a4b88665a16c9f078           321             1        43.62   
4  0004aac84e0df4da2b147fca70cf8255           288             1       196.89   

   avg_review_score  churn_label  
0               5.0            0  
1               4.0            0  
2               3.0            1  
3               4.0            1  
4               5.0            1  

--- Model Evaluation ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     11340
           1       1.00      1.00      1.00     16484

    accuracy                           1.00     27824
   macro avg       1.0

In [11]:
# --- STEP 7: RISK SEGMENTATION ---
# Create segments based on probability for the Dashboard
def assign_risk(prob):
    if prob > 0.7:
        return 'High Risk'
    elif prob > 0.3:
        return 'Medium Risk'
    else:
        return 'Low Risk'

customer_data['risk_segment'] = customer_data['churn_probability'].apply(assign_risk)

In [13]:
# --- STEP 8: EXPORT CLEAN DATA FOR POWER BI ---
# This file will be loaded into Power BI
output_filename = 'output/ecommerce_churn_processed_data.csv'
customer_data.to_csv(output_filename, index=False)

print(f"\nProcessing Complete! File saved as: {output_filename}")


Processing Complete! File saved as: output/ecommerce_churn_processed_data.csv
